# 05 · Classifying planets vs false positives

The pipeline finds signals and this notebook decides **which signals look like planets.** It trains a classifier on the vetting features from notebook 04, tests whether it learned real physics or a shortcut, finds and fixes problems in its own training labels, and then uses it on **real unconfirmed TESS candidates**.

1. **The dataset**: 933 labelled stars from notebook 03
2. **Exploring the features**: which separate the classes on their own?
3. **First models**: a random forest, with and without signal-strength features
4. **What the model relies on**: permutation importance
5. **Errors and label noise**: the most confident mistakes reveal mislabelled training data
6. **Scoring real candidates**: ranking 403 unconfirmed TESS signals

This is **supervised learning**: each training example has a label (planet or false positive) from astronomers' follow-up observations, and the model learns to predict it from the features.

## Setup

In [1]:
%load_ext autoreload
%autoreload 2
import warnings
warnings.filterwarnings("ignore", message=".*oktopus.*")

import sys
sys.path.append("..")
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix
from sklearn.inspection import permutation_importance

from src.batch import DATA_DIR, load_toi_catalog, run_batch, classify_match
from src.features import FEATURE_COLUMNS, DERIVED_COLUMNS, add_derived_features

RERUN = False   # True = reprocess the candidate batch (~8 hours); False = load saved results

def make_model():
    """The random forest used throughout, so every comparison uses identical settings."""
    return RandomForestClassifier(n_estimators=500, min_samples_leaf=3,
                                  class_weight="balanced", random_state=42, n_jobs=-1)

def clean_X(frame, cols):
    """Feature matrix with any infinite values (from divisions) turned into missing values."""
    return frame[cols].replace([np.inf, -np.inf], np.nan)

Settings, used everywhere below:

- **500 trees**, each a set of rules like "if `shape_ratio` < 0.6 and `duty_cycle` > 0.1, lean false positive". The forest basically votes
- **`min_samples_leaf=3`**: no rule may single out fewer than 3 stars, so the trees can't memorize individual examples
- **`class_weight="balanced"`**: stops the model favouring the larger class (false positives)

---
# Part 1: The dataset

In [2]:
results = pd.read_csv(DATA_DIR / "train_v1" / "results.csv")
df = results[results["status"] == "ok"].copy()
df = add_derived_features(df)
features = FEATURE_COLUMNS + DERIVED_COLUMNS
y = (df["label"] == "planet").astype(int)  # 1 = planet, 0 = false positive

print(len(df), "rows,", len(features), "features")
df["label"].value_counts()

933 rows, 19 features


label
false_positive    515
planet            418
Name: count, dtype: int64

`add_derived_features` adds three features from each star's catalog radius and surface gravity: the planet's implied radius, the transit duration expected for that star, and the ratio of actual to expected duration.

---
# Part 2: Exploring the features

## Medians by class

In [3]:
df.groupby("label")[features].median().T

label,false_positive,planet
duty_cycle,0.040418,0.026331
depth_odd_ppm,1143.964082,4180.208298
depth_even_ppm,1194.318695,4362.288797
odd_even_sigma,1.263754,1.091451
secondary_depth_ppm,8.778531,8.879525
secondary_sigma,0.161589,0.189831
secondary_ratio,0.005675,0.002506
shape_ratio,0.720762,0.865025
red_noise_beta,1.175710,1.082245
snr_red,19.594442,63.254618


The biggest gaps are in **depth and SNR**: planets' median depth is about 3.5× larger, and `snr_red` about 3× higher. But that's suspicious. Eclipsing binaries are two *stars*, so their dips should often be **deeper** than a planet's.

The explanation is **confirmation bias** in the catalog: astronomers confirm the easy planets first (big, deep, strong signals), while many false positives are faint, diluted background binaries. A model leaning on "strong signal = planet" would look good here and fail on unconfirmed candidates, which tend to be weaker. Part 3 tests for exactly this.

## How well does each feature separate the classes on its own?

**AUC** is the probability that a random planet has a higher value than a random false positive. 0.5 means useless; the further from 0.5 in *either* direction, the better (below 0.5 means false positives tend to be higher).

In [4]:
rows = []
for feat in features:
    has_value = df[feat].notna()
    auc = roc_auc_score(y[has_value], df.loc[has_value, feat])
    rows.append({"feature": feat, "auc": auc, "n_rows": int(has_value.sum())})

feature_auc = pd.DataFrame(rows)
feature_auc["separation"] = (feature_auc["auc"] - 0.5).abs()
feature_auc.sort_values("separation", ascending=False)

,feature,auc,n_rows,separation
7,shape_ratio,0.718456,932,0.218456
9,snr_red,0.690016,926,0.190016
1,depth_odd_ppm,0.664161,909,0.164161
2,depth_even_ppm,0.662436,922,0.162436
0,duty_cycle,0.358935,933,0.141065
8,red_noise_beta,0.381026,926,0.118974
18,duration_ratio,0.583114,807,0.083114
13,centroid_offset_px,0.575575,933,0.075575
15,centroid_source_offset_px,0.434300,932,0.065700
17,expected_duration_days,0.436961,807,0.063039


- **`shape_ratio` ranks first (≈0.72)**, ahead of depth and SNR: the U-vs-V difference is the strongest single physical clue.
- **Directions match the physics:** false positives have longer duty cycles, more correlated noise, and dip sources farther from the target.
- **The centroid pair shows why scaling matters.** The raw offset points the *wrong* way (deeper planets nudge the centroid more); dividing by depth flips it to the physically sensible direction.
- **No single feature is great.** Each catches one kind of impostor; the model's job is to combine them.

`odd_even_sigma` and `secondary_sigma` look weak (near 0.5), but AUC averages over *all* stars, and only some false positives are binaries at half their period. The information is in the **tail**:

In [5]:
for feat in ["odd_even_sigma", "secondary_sigma"]:
    frac = df.groupby("label")[feat].apply(lambda s: (s > 5).mean())
    print(feat, "> 5 sigma:", frac.round(3).to_dict())

odd_even_sigma > 5 sigma: {'false_positive': 0.132, 'planet': 0.045}
secondary_sigma > 5 sigma: {'false_positive': 0.068, 'planet': 0.031}


Odd/even differences above 5σ are about **3× more common** among false positives. Rare but meaningful: a tree-based model can use exactly that kind of rule.

---
# Part 3: First models

Two random forests:

- **All features**
- **No signal strength**: without the depths, `snr_red`, the depth-derived planet radius, and the raw centroid offset

If the second is nearly as good, the model has learned physics rather than the "strong signal = planet" shortcut.

**5-fold cross-validation**: the data is split into 5 parts; each part is predicted by a model trained on the other 4, so every star gets a score from a model that **never saw it**. "Stratified" keeps the planet/false-positive mix the same in every part.

In [6]:
strength = ["depth_odd_ppm", "depth_even_ppm", "secondary_depth_ppm",
            "snr_red", "planet_radius_re", "centroid_offset_px"]
feature_sets = {
    "all features": features,
    "no signal strength": [f for f in features if f not in strength],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
preds = {}
for name, cols in feature_sets.items():
    p = cross_val_predict(make_model(), clean_X(df, cols), y, cv=cv, method="predict_proba")[:, 1]
    preds[name] = p
    print(f"{name}: ROC AUC {roc_auc_score(y, p):.3f}   PR AUC {average_precision_score(y, p):.3f}")
    print(confusion_matrix(y, p >= 0.5), "\n")

all features: ROC AUC 0.932   PR AUC 0.921
[[446  69]
 [ 65 353]] 

no signal strength: ROC AUC 0.913   PR AUC 0.893
[[429  86]
 [ 76 342]] 



- **ROC AUC**: overall ranking quality. Compare with the best single feature (~0.72): combining features is what makes it work.
- **PR AUC**: how clean the "planet" predictions are. Random guessing scores ≈ 0.45 (the share of planets).
- **Confusion matrix**: rows are the truth (false positive, planet), columns the prediction. The top-right cell counts false positives wrongly called planets.

**Removing the signal-strength features costs only about 0.02.** Most of what the model learned is physics.

## Does it still work on weak signals?

Unconfirmed candidates tend to be weaker, so split the stars at the median `snr_red` and score each half separately:

In [7]:
weak = (df["snr_red"] < df["snr_red"].median()).values
print(f"median snr_red: {df['snr_red'].median():.1f}\n")

for name, p in preds.items():
    print(f"{name}: weak signals {roc_auc_score(y[weak], p[weak]):.3f}   "
          f"strong signals {roc_auc_score(y[~weak], p[~weak]):.3f}")

print("\nplanet share - weak:", round(y[weak].mean(), 3), "  strong:", round(y[~weak].mean(), 3))

median snr_red: 26.2

all features: weak signals 0.928   strong signals 0.929
no signal strength: weak signals 0.897   strong signals 0.920

planet share - weak: 0.324   strong: 0.57


- **The all-features model holds up equally well on weak signals.** It separates the classes *within* each half, so it isn't just sorting by signal strength.
- **The physics-only model drops slightly on weak signals**, as expected: subtle features like exact shape are harder to measure when the signal is faint.
- **The planet share shows the confirmation bias directly:** planets are a minority among weaker signals.

One limit: "weak" here means `snr_red` below about 26, still a clear signal. Candidates weaker than anything in this training data need extra caution (Part 6 flags them).

---
# Part 4: What the model relies on

**Permutation importance**: shuffle one feature's values across the stars, breaking its link to the labels, and measure how much the AUC drops. A big drop means the model depends on that feature. Measured on 25% of stars held out from training.

In [8]:
X = clean_X(df, features)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

model = make_model().fit(X_tr, y_tr)
base_auc = roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])
print(f"held-out ROC AUC: {base_auc:.3f}")

imp = permutation_importance(model, X_te, y_te, scoring="roc_auc", n_repeats=20, random_state=42, n_jobs=-1)
importance = (pd.DataFrame({"feature": features, "importance": imp.importances_mean, "std": imp.importances_std})
              .sort_values("importance", ascending=False))
importance

held-out ROC AUC: 0.918


,feature,importance,std
7,shape_ratio,0.057918,0.015738
0,duty_cycle,0.029675,0.008674
16,planet_radius_re,0.021510,0.004128
9,snr_red,0.018391,0.005800
17,expected_duration_days,0.014212,0.005013
18,duration_ratio,0.012252,0.007433
2,depth_even_ppm,0.006445,0.003198
6,secondary_ratio,0.006072,0.003056
4,secondary_depth_ppm,0.005113,0.002050
11,max_transit_frac,0.004404,0.001993


- **`shape_ratio` is the most important single feature**, followed by the timing features (`duty_cycle`, expected and actual duration): whether the transit lasts as long as a planet around *this* star should.
- **`planet_radius_re` ranks highly despite a single-feature AUC of ~0.51.** Weak alone, strong in combination: "large *and* V-shaped" is a binary signature a single-feature test can't see.
- **The depth features rank low individually, which is misleading.** They carry overlapping information, so shuffling one lets the model fall back on another.

Test that by shuffling the whole depth group **together**:

In [9]:
depth_group = ["depth_odd_ppm", "depth_even_ppm", "planet_radius_re", "snr_red"]
rng = np.random.default_rng(42)

drops = []
for _ in range(20):
    X_shuf = X_te.copy()
    order = rng.permutation(len(X_shuf))
    for col in depth_group:
        X_shuf[col] = X_te[col].values[order] # same reshuffle for all four
    drops.append(base_auc - roc_auc_score(y_te, model.predict_proba(X_shuf)[:, 1]))

print(f"Depth group, shuffled together: AUC drop {np.mean(drops):.3f} ± {np.std(drops):.3f}")
print(f"shape_ratio alone: AUC drop {importance.set_index('feature').loc['shape_ratio', 'importance']:.3f}")

Depth group, shuffled together: AUC drop 0.083 ± 0.015
shape_ratio alone: AUC drop 0.058


Together, depth information is the **biggest** thing this model relies on, bigger than its parts added up (the signature of redundancy).

That looks like a contradiction with Part 3, where removing depth cost only 0.02. Both are true, and they answer different questions:

- **Permutation importance:** how much does *this trained model* rely on these features? A lot: depth is an easy shortcut in this data.
- **Retraining without them:** how much does a model *need* them? Very little: the physics features can do almost the same job.

That's why the physics-only model leads the candidate ranking in Part 6.

---
# Part 5: Errors and label noise

Each star's cross-validated score came from a model that never saw it. The most confident mistakes:

In [10]:
df["p_all"] = preds["all features"]
df["p_physics"] = preds["no signal strength"]
cols = ["target", "label", "p_all", "p_physics", "period_days", "depth_ppm",
        "snr_red", "shape_ratio", "duty_cycle", "odd_even_sigma"]

print("Planets the model is most sure are NOT planets:")
display(df[df["label"] == "planet"].nsmallest(8, "p_all")[cols])
print("False positives the model is most sure ARE planets:")
display(df[df["label"] == "false_positive"].nlargest(8, "p_all")[cols])

Planets the model is most sure are NOT planets:


,target,label,p_all,p_physics,period_days,depth_ppm,snr_red,shape_ratio,duty_cycle,odd_even_sigma
54,TIC 201175570,planet,0.111243,0.193883,0.561878,60.547827,2.419592,0.959822,0.355949,1.305537
64,TIC 377780790,planet,0.114540,0.068966,0.837574,171.853259,4.929862,0.709367,0.128944,1.554869
349,TIC 328934463,planet,0.140423,0.142563,0.533105,64.668702,-0.140384,NaN,0.375161,0.950826
283,TIC 17865622,planet,0.153122,0.146213,3.119960,7720.506562,95.523956,0.679437,0.016667,2.125969
44,TIC 186812530,planet,0.181257,0.290370,3.677188,3462.276741,84.968291,0.697173,0.014141,0.947845
178,TIC 348835438,planet,0.198946,0.188407,6.203468,1380.599492,24.903143,1.034044,0.032240,1.277500
233,TIC 383390264,planet,0.200200,0.235529,1.164597,2424.158073,8.678094,0.883118,0.171733,41.409276
39,TIC 172518755,planet,0.204561,0.254267,3.208878,2497.575705,72.166571,0.682539,0.019945,1.792967


False positives the model is most sure ARE planets:


,target,label,p_all,p_physics,period_days,depth_ppm,snr_red,shape_ratio,duty_cycle,odd_even_sigma
696,TIC 294179389,false_positive,0.906906,0.922554,2.679328,7451.116594,99.416800,0.845521,0.040309,0.155276
480,TIC 95010343,false_positive,0.862496,0.872120,6.878215,556.845541,11.527529,0.825063,0.017446,1.553692
769,TIC 446549905,false_positive,0.842382,0.910303,2.454564,3953.685628,96.838324,0.828244,0.039111,1.270275
869,TIC 34551972,false_positive,0.841272,0.782808,4.854268,7907.636108,150.099394,0.890445,0.041201,0.460976
474,TIC 394509844,false_positive,0.821216,0.867681,6.892232,5847.798441,97.996469,0.878919,0.027277,1.119541
1065,TIC 219338557,false_positive,0.815270,0.702859,8.197796,948.243972,14.295632,0.722150,0.011710,0.148194
1039,TIC 388624270,false_positive,0.803227,0.776365,14.253561,7141.430938,83.993213,0.781953,0.014032,0.153819
876,TIC 405904232,false_positive,0.797278,0.630638,11.076407,4666.194344,94.498429,0.882828,0.016973,0.485552


**The "missed planets" are mostly not model errors.** Several have `snr_red` near zero: the pipeline never found the planet and measured noise instead, which the model correctly rejected. One has `odd_even_sigma` ≈ 41, the signature of the pipeline finding *half* the planet's period. But the label still says "planet." That's **label noise**: the label describes the catalog's signal, while the features describe whatever the pipeline found. A few genuine misses are strong, V-shaped signals, likely **grazing** planets that only clip their star's edge.

**The false positives the model calls planets look like textbook planets**, many with ~0.7–0.8% depths: the size of a hot Jupiter, but also of a **brown dwarf or small star**, which are about Jupiter's size. Only a mass measurement (follow-up spectroscopy) can tell them apart. That's a genuine ceiling for any light-curve classifier.

## How much label noise?

In [11]:
df["period_match"] = [classify_match(p, pc) for p, pc in zip(df["period_days"], df["catalog_period"])]
pd.crosstab(df["label"], df["period_match"])

period_match,alias,match,miss
label,,,
false_positive,42,389,84
planet,3,382,33


About **36 "planets"** (misses and aliases) describe a signal other than the planet. For false positives, whatever the pipeline found is still not a confirmed planet, so their labels stay broadly correct.

## Retraining on clean labels

Keep only planets whose catalog period was recovered, and all false positives. Then evaluate two ways: on everything, and on **matched signals only**, the honest test of the real question: *once the pipeline has found the true signal, can the model tell planet from impostor?*

In [12]:
clean = df[(df["label"] == "false_positive") | (df["period_match"] == "match")].copy()
y_c = (clean["label"] == "planet").astype(int)
matched = (clean["period_match"] == "match").values
print(len(clean), "rows:", y_c.sum(), "planets,", (1 - y_c).sum(), "false positives\n")

for name, cols in feature_sets.items():
    p = cross_val_predict(make_model(), clean_X(clean, cols), y_c, cv=cv, method="predict_proba")[:, 1]
    print(f"{name}:")
    print(f"   all cleaned rows   ROC AUC {roc_auc_score(y_c, p):.3f}")
    print(f"   matched signals    ROC AUC {roc_auc_score(y_c[matched], p[matched]):.3f}")

897 rows: 382 planets, 515 false positives

all features:
   all cleaned rows   ROC AUC 0.949
   matched signals    ROC AUC 0.939
no signal strength:
   all cleaned rows   ROC AUC 0.933
   matched signals    ROC AUC 0.919


**Cleaning the labels improved both models by about 0.02.** Mislabelled examples were teaching the model wrong lessons.

The system's performance now splits into two honest stages:

1. **Detection:** the pipeline recovers about **90%** of confirmed planets (notebooks 03 and 06)
2. **Classification:** among recovered signals, the model separates planets from false positives with **~0.94 ROC AUC** (~0.92 with physics features only)

---
# Part 6: Scoring real candidates

The point of all the testing: say something useful about stars **nobody has an answer for yet.** The catalog's ~4,800 **planet candidates (PC)** are TESS signals awaiting follow-up. Applying the same filters as the training set leaves 691.

In [13]:
toi = load_toi_catalog()
counts = toi["tid"].value_counts()
pc = toi[(toi["tfopwg_disp"] == "PC")
         & (toi["tid"].map(counts) == 1)
         & toi["pl_orbper"].between(0.5, 15.0)
         & (toi["st_tmag"] <= 11.0)].copy()
pc["label"] = "candidate"
pc["target"] = "TIC " + pc["tid"].astype(int).astype(str)
print(len(pc), "candidates")

saved = DATA_DIR / "candidates_v1" / "results.csv"
cand = run_batch(pc, name="candidates_v1", max_sectors=3) if (RERUN or not saved.exists()) else pd.read_csv(saved)
print(cand["status"].value_counts())

691 candidates
status
ok       651
error     40
Name: count, dtype: int64


## Only score what can be trusted

Three gates, so the model only scores signals that really are the catalog's candidate:

1. **The pipeline recovered the catalog period** (the same check used to clean the training labels)
2. **`snr_red` ≥ 7**: a real detection
3. **At least 3 transits**: not a single event

In [14]:
cand = cand[cand["status"] == "ok"].copy()
cand = add_derived_features(cand)
cand["period_match"] = [classify_match(p, pc_) for p, pc_ in zip(cand["period_days"], cand["catalog_period"])]
print(cand["period_match"].value_counts(), "\n")

usable = cand[(cand["period_match"] == "match")
              & (cand["snr_red"] >= 7)
              & (cand["n_transits"] >= 3)].copy()

snr_floor = clean["snr_red"].quantile(0.05)   # weakest 5% of training signals
usable["weaker_than_training"] = usable["snr_red"] < snr_floor
print(len(usable), "candidates scored")

period_match
match    447
miss     184
alias     20
Name: count, dtype: int64 

403 candidates scored


Only about two-thirds of candidates match their catalog period, lower than for confirmed planets: candidates tend to be **weaker**, because the easy ones were resolved long ago. The ranking covers the clearer signals; misses aren't evidence *against* a candidate.

## Train the final models and rank

Cross-validation was for **measuring** performance. The final models train on **all** 897 cleaned stars, then score the candidates. The **physics-only score leads the ranking**, so the confirmation bias isn't carried onto weaker candidates.

In [15]:
short = {"all features": "all", "no signal strength": "physics"}
for name, cols in feature_sets.items():
    final_model = make_model().fit(clean_X(clean, cols), y_c)
    usable[f"score_{short[name]}"] = final_model.predict_proba(clean_X(usable, cols))[:, 1]

usable["too_big"] = usable["planet_radius_re"] > 22   # ~2 Jupiter radii: beyond almost all planets
candidate_ranking = usable.sort_values("score_physics", ascending=False)
candidate_ranking.to_csv(DATA_DIR / "candidate_ranking_v1.csv", index=False)

top_cols = ["target", "toi", "period_days", "depth_ppm", "st_rad", "planet_radius_re",
            "snr_red", "score_physics", "score_all", "too_big", "weaker_than_training"]
candidate_ranking[top_cols].head(20)

,target,toi,period_days,depth_ppm,st_rad,planet_radius_re,snr_red,score_physics,score_all,too_big,weaker_than_training
175,TIC 470127886,2031.01,5.715353,12219.947609,1.184440,14.284745,189.120707,0.983831,0.998133,False,False
11,TIC 146172354,1036.01,3.779532,2741.924718,1.180000,6.741158,61.386401,0.978251,0.845787,False,False
8,TIC 341420329,1019.01,5.234292,20501.366976,1.562030,24.400882,278.594473,0.976028,0.772873,True,False
150,TIC 186936449,2665.01,2.845648,2683.158600,1.300000,7.346683,49.730702,0.967649,0.761129,False,False
91,TIC 257241363,1718.01,5.587047,1830.747625,0.940000,4.388002,82.903498,0.963410,0.847092,False,False
62,TIC 53875855,1491.01,4.490281,2506.080294,1.230000,6.717804,43.555355,0.961726,0.803267,False,False
488,TIC 160432093,5971.01,3.991436,9081.789135,1.751650,18.212006,136.631375,0.959080,0.962931,False,False
574,TIC 182405015,6105.01,3.112300,14025.969204,1.120740,14.480918,220.036714,0.956819,0.962259,False,False
243,TIC 203377303,3493.01,8.159484,743.183354,1.211510,3.603294,30.466684,0.955871,0.830284,False,False
7,TIC 114018671,1011.01,2.470508,262.745706,0.940000,1.662342,19.773651,0.951687,0.878674,False,False


**The top of the ranking** mixes hot Jupiters (~12–18 Earth radii) with much smaller planets, including candidates under 2 Earth radii. The star's radius matters as much as the depth: the same dip means a far smaller planet around a small red dwarf.

**Treat deep, top-ranked candidates with care.** That's the brown-dwarf territory from Part 5, where light curves can't decide. TOI 1019.01 is flagged `too_big` (~2.2 Jupiter radii), and the all-features model already scored it lower: two independent clues pointing the same way.

## The other end, and the triage

In [16]:
bottom_cols = ["target", "toi", "period_days", "depth_ppm", "planet_radius_re", "shape_ratio",
               "odd_even_sigma", "duty_cycle", "score_physics", "score_all"]
display(candidate_ranking[bottom_cols].tail(10))

print("Physics score > 0.8:", (candidate_ranking["score_physics"] > 0.8).sum())
print("Physics score < 0.2:", (candidate_ranking["score_physics"] < 0.2).sum())
print("In between:", candidate_ranking["score_physics"].between(0.2, 0.8).sum())

,target,toi,period_days,depth_ppm,planet_radius_re,shape_ratio,odd_even_sigma,duty_cycle,score_physics,score_all
123,TIC 234160911,2600.01,5.343029,2387.293886,18.657169,0.566808,1.412257,0.017967,0.051042,0.034287
431,TIC 411930845,5109.01,3.209975,1678.294266,21.051338,0.658073,1.279260,0.026168,0.046579,0.059914
575,TIC 133334108,637.01,2.857961,457.109023,NaN,0.743474,1.082542,0.069980,0.045788,0.046114
532,TIC 238061845,579.01,1.684186,969.546338,16.781701,0.745152,1.189894,0.045126,0.044272,0.062330
342,TIC 450649506,4371.01,1.058883,1220.882642,6.175565,0.638072,0.462251,0.090662,0.044131,0.017400
227,TIC 69819610,2324.01,1.037171,311.956391,2.890433,0.535928,2.849044,0.169692,0.038436,0.061707
197,TIC 229888286,2057.01,5.597937,2325.926246,9.667352,0.724299,1.458547,0.035727,0.034980,0.038529
120,TIC 229167359,2594.01,2.681848,1368.401575,21.309117,0.532462,0.675125,0.019390,0.032992,0.036387
428,TIC 19612445,4837.01,4.464512,2185.970947,8.671533,0.706965,2.114639,0.008960,0.029599,0.044608
469,TIC 146406806,594.01,13.951003,7617.478577,NaN,0.650590,2.524784,0.009462,0.026562,0.048144


Physics score > 0.8: 76
Physics score < 0.2: 101
In between: 226


The bottom candidates are mostly **V-shaped** (low `shape_ratio`) and often **too large** for planets. Their odd/even values are unremarkable: these were found at their catalog period, so only one eclipse is visible, and shape and size give them away instead.

**The triage** is the most useful way to present the result:

- **Score > 0.8: high priority** for follow-up
- **Score < 0.2: low priority**, likely impostors
- **In between: needs more data**

**Read scores as a ranking, not literal probabilities.** The model trained on a roughly balanced set, so 0.9 doesn't mean "a 90% chance of being a planet" among real candidates. Turning scores into true probabilities (**calibration**) is a possible extension.

---
# Summary

- **Classifier:** 0.94 ROC AUC on recovered signals (5-fold CV), 0.92 with physics features only, holding up on weak signals
- **Most important evidence:** transit shape and timing, plus depth as a strong (but bias-prone) shortcut
- **Found and fixed:** label noise from pipeline misses, improving both models
- **A genuine limit:** brown dwarfs and small stars mimic hot Jupiters in light curves
- **Output:** 403 unconfirmed TESS candidates ranked and triaged, saved to `data/candidate_ranking_v1.csv`

These are **rankings of existing candidates, not discoveries.** As astronomers resolve them over time, the predictions can be checked against their verdicts: a genuine test on unseen data.